In [1]:
import os
import re
import json
import random
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


In [2]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

MODEL_TYPE = "bigru"

Device: cpu


In [3]:
TARGET_PATH = "/content/boun_target_train.txt"

In [4]:
SAVE_MODEL_PATH = f"/content/{MODEL_TYPE}_char_completion.pt"
SAVE_VOCAB_PATH = "/content/vocab.json"

In [5]:
MAX_WORDS = 10
MASK_PROB_WORD = 0.90       # bir kelimenin bozulma olasılığı
MASK_PROB_CHAR = 1.0        # bozulan kelimede kaç karakter bozulacağı mantığı burada 1 karakter
MIN_SENT_LEN = 2
MAX_SENT_CHAR_LEN = 300

BATCH_SIZE = 64
EPOCHS = 5
LR = 0.001
EMB_DIM = 128
HIDDEN_DIM = 256

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

In [6]:
# ------------------------------------------------------------
# 2) METİN TEMİZLEME VE BOZMA
# ------------------------------------------------------------
def normalize_target(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ\s\.\,\!\?\-']", " ", text)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_input(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ_\s\.\,\!\?\-']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_to_max_words(sentence: str, max_words: int = 10):
    words = sentence.split()
    if len(words) <= max_words:
        return [sentence]

    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i+max_words]).strip()
        if chunk:
            chunks.append(chunk)
    return chunks

def corrupt_word(word: str, corruption_prob: float = 0.70) -> str:
    if len(word) <= 1:
        return word

    if random.random() > corruption_prob:
        return word

    idx = random.randint(0, len(word) - 1)
    word_list = list(word)
    word_list[idx] = "_"
    return "".join(word_list)

def corrupt_sentence(sentence: str) -> str:
    words = sentence.split()
    corrupted_words = [corrupt_word(w, MASK_PROB_WORD) for w in words]
    corrupted = " ".join(corrupted_words)

    # hiçbir şey bozulmadıysa en az bir kelime boz
    if corrupted == sentence and len(words) > 0:
        i = random.randint(0, len(words) - 1)
        words[i] = corrupt_word(words[i], corruption_prob=1.0)
        corrupted = " ".join(words)

    return corrupted

def build_pairs_from_target(target_path: str):
    with open(target_path, "r", encoding="utf8") as f:
        lines = [line.strip() for line in f if line.strip()]

    # normalize
    lines = [normalize_target(line) for line in lines]
    lines = [line for line in lines if line]

    # duplicate temizliği
    lines = list(dict.fromkeys(lines))

    pairs = []
    for line in lines:
        chunks = split_to_max_words(line, MAX_WORDS)

        for chunk in chunks:
            chunk = chunk.strip()

            if len(chunk) < MIN_SENT_LEN:
                continue
            if len(chunk) > MAX_SENT_CHAR_LEN:
                continue

            corrupted = corrupt_sentence(chunk)
            pairs.append((corrupted, chunk))

    # input-target duplicate pair temizliği
    pairs = list(dict.fromkeys(pairs))
    return pairs

In [7]:
# ------------------------------------------------------------
# 3) VERİYİ HAZIRLA
# ------------------------------------------------------------
DEV_TARGET_PATH = "/content/boun_target_dev.txt"
TEST_TARGET_PATH = "/content/boun_target_test.txt"

train_pairs = build_pairs_from_target(TARGET_PATH)
print("Train örnek sayısı:", len(train_pairs))
print("İlk 5 örnek:")
for i in range(min(5, len(train_pairs))):
    print("INP:", train_pairs[i][0])
    print("TGT:", train_pairs[i][1])
    print("-" * 50)

dev_pairs = build_pairs_from_target(DEV_TARGET_PATH) if DEV_TARGET_PATH else []
test_pairs = build_pairs_from_target(TEST_TARGET_PATH) if TEST_TARGET_PATH else []

# train/dev split yoksa train içinden küçük bir validation ayıralım
if not dev_pairs:
    random.shuffle(train_pairs)
    split_idx = int(len(train_pairs) * 0.9)
    dev_pairs = train_pairs[split_idx:]
    train_pairs = train_pairs[:split_idx]

print("Train final:", len(train_pairs))
print("Dev final:", len(dev_pairs))
print("Test final:", len(test_pairs))


Train örnek sayısı: 10782
İlk 5 örnek:
INP: _mekli a_bay y_ldırım taşyumru_ bir_z _a mesl_ği gere_i ömr_ bo_u
TGT: emekli albay yıldırım taşyumruk biraz da mesleği gereği ömrü boyu
--------------------------------------------------
INP: _ert _e otori_er _ir baba o_muştur.
TGT: sert ve otoriter bir baba olmuştur.
--------------------------------------------------
INP: ar_cı pa_k yerinde_ ger_ al_ak d_ ço_ kol_y... _kıllı t_lefondaki
TGT: aracı park yerinden geri almak da çok kolay... akıllı telefondaki
--------------------------------------------------
INP: uygulam_ üzeri_den sürüc_ler ar_cı _e z_man t_slim alm_k istediği_i bildi_iyor.
TGT: uygulama üzerinden sürücüler aracı ne zaman teslim almak istediğini bildiriyor.
--------------------------------------------------
INP: _u _alışmaların devamı_da _üm mezarlıklarımızı kapsa_an çalışma_ar yapılac_k _e b_
TGT: bu çalışmaların devamında tüm mezarlıklarımızı kapsayan çalışmalar yapılacak ve bu
---------------------------------------------

In [8]:
# ------------------------------------------------------------
# 4) VOCAB
# ------------------------------------------------------------
all_texts = []
for x, y in train_pairs + dev_pairs + test_pairs:
    all_texts.append(x)
    all_texts.append(y)

chars = set()
for text in all_texts:
    chars.update(list(text))

chars = sorted(list(chars))
itos = [PAD_TOKEN, UNK_TOKEN] + chars
stoi = {ch: i for i, ch in enumerate(itos)}

with open(SAVE_VOCAB_PATH, "w", encoding="utf8") as f:
    json.dump(stoi, f, ensure_ascii=False, indent=2)

print("Vocab size:", len(itos))

def encode(text: str):
    return [stoi.get(ch, stoi[UNK_TOKEN]) for ch in text]

def decode(ids):
    out = []
    for idx in ids:
        ch = itos[idx]
        if ch not in [PAD_TOKEN, UNK_TOKEN]:
            out.append(ch)
    return "".join(out)

max_len = 0
for x, y in train_pairs + dev_pairs + test_pairs:
    max_len = max(max_len, len(x), len(y))

print("Max sequence length:", max_len)

def pad(seq):
    seq = seq[:]
    while len(seq) < max_len:
        seq.append(stoi[PAD_TOKEN])
    return seq

Vocab size: 42
Max sequence length: 119


In [9]:
# ------------------------------------------------------------
# 5) DATASET
# ------------------------------------------------------------
class CharDataset(Dataset):
    def __init__(self, pairs):
        self.inputs = []
        self.targets = []

        for inp, out in pairs:
            self.inputs.append(torch.tensor(pad(encode(inp)), dtype=torch.long))
            self.targets.append(torch.tensor(pad(encode(out)), dtype=torch.long))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

train_ds = CharDataset(train_pairs)
dev_ds = CharDataset(dev_pairs)
test_ds = CharDataset(test_pairs) if len(test_pairs) > 0 else None

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False) if test_ds else None



In [10]:
# ------------------------------------------------------------
# 6) MODEL
# ------------------------------------------------------------
class CharCompletionModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, model_type="bigru"):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            emb_dim,
            padding_idx=stoi[PAD_TOKEN]
        )

        if model_type == "rnn":
            self.rnn = nn.RNN(
                input_size=emb_dim,
                hidden_size=hidden_dim,
                batch_first=True,
                bidirectional=False
            )
            rnn_output_dim = hidden_dim

        elif model_type == "gru":
            self.rnn = nn.GRU(
                input_size=emb_dim,
                hidden_size=hidden_dim,
                batch_first=True,
                bidirectional=False
            )
            rnn_output_dim = hidden_dim

        elif model_type == "lstm":
            self.rnn = nn.LSTM(
                input_size=emb_dim,
                hidden_size=hidden_dim,
                batch_first=True,
                bidirectional=False
            )
            rnn_output_dim = hidden_dim

        elif model_type == "birnn":
            self.rnn = nn.RNN(
                input_size=emb_dim,
                hidden_size=hidden_dim,
                batch_first=True,
                bidirectional=True
            )
            rnn_output_dim = hidden_dim * 2

        elif model_type == "bilstm":
            self.rnn = nn.LSTM(
                input_size=emb_dim,
                hidden_size=hidden_dim,
                batch_first=True,
                bidirectional=True
            )
            rnn_output_dim = hidden_dim * 2

        else:  # bigru
            self.rnn = nn.GRU(
                input_size=emb_dim,
                hidden_size=hidden_dim,
                batch_first=True,
                bidirectional=True
            )
            rnn_output_dim = hidden_dim * 2

        self.fc = nn.Linear(rnn_output_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

model = CharCompletionModel(
    vocab_size=len(itos),
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    model_type=MODEL_TYPE
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=stoi[PAD_TOKEN])
optimizer = optim.Adam(model.parameters(), lr=LR)


In [11]:
# ------------------------------------------------------------
# 7) METRİKLER
# ------------------------------------------------------------
def char_accuracy(logits, y):
    preds = logits.argmax(dim=-1)
    mask = (y != stoi[PAD_TOKEN])

    correct = ((preds == y) & mask).sum().item()
    total = mask.sum().item()

    return correct / total if total > 0 else 0.0

def sentence_accuracy(logits, y):
    preds = logits.argmax(dim=-1)
    mask = (y != stoi[PAD_TOKEN])

    correct_sent = 0
    total_sent = y.size(0)

    for i in range(y.size(0)):
        pred_seq = preds[i][mask[i]]
        true_seq = y[i][mask[i]]
        if torch.equal(pred_seq, true_seq):
            correct_sent += 1

    return correct_sent / total_sent if total_sent > 0 else 0.0

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    total_char_acc = 0
    total_sent_acc = 0

    with torch.no_grad():
        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            Y_batch = Y_batch.to(DEVICE)

            output = model(X_batch)
            loss = criterion(output.view(-1, len(itos)), Y_batch.view(-1))

            total_loss += loss.item()
            total_char_acc += char_accuracy(output, Y_batch)
            total_sent_acc += sentence_accuracy(output, Y_batch)

    n = len(loader)
    return total_loss / n, total_char_acc / n, total_sent_acc / n


In [12]:
# ------------------------------------------------------------
# 8) EĞİTİM
# ------------------------------------------------------------
best_dev_loss = math.inf

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    total_char_acc = 0
    total_sent_acc = 0

    for X_batch, Y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        Y_batch = Y_batch.to(DEVICE)

        optimizer.zero_grad()
        output = model(X_batch)

        loss = criterion(output.view(-1, len(itos)), Y_batch.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_char_acc += char_accuracy(output, Y_batch)
        total_sent_acc += sentence_accuracy(output, Y_batch)

    train_loss = total_loss / len(train_loader)
    train_char_acc = total_char_acc / len(train_loader)
    train_sent_acc = total_sent_acc / len(train_loader)

    dev_loss, dev_char_acc, dev_sent_acc = evaluate(model, dev_loader)

    print(f"Epoch {epoch:02d}")
    print(f"Train Loss: {train_loss:.4f} | Train Char Acc: {train_char_acc:.4f} | Train Sent Acc: {train_sent_acc:.4f}")
    print(f"Dev   Loss: {dev_loss:.4f} | Dev   Char Acc: {dev_char_acc:.4f} | Dev   Sent Acc: {dev_sent_acc:.4f}")
    print("-" * 80)

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "stoi": stoi,
            "itos": itos,
            "model_type": MODEL_TYPE,
            "emb_dim": EMB_DIM,
            "hidden_dim": HIDDEN_DIM,
            "max_len": max_len
        }, SAVE_MODEL_PATH)
        print("En iyi model kaydedildi:", SAVE_MODEL_PATH)


Epoch 01
Train Loss: 0.4604 | Train Char Acc: 0.8910 | Train Sent Acc: 0.0179
Dev   Loss: 0.2212 | Dev   Char Acc: 0.9331 | Dev   Sent Acc: 0.0418
--------------------------------------------------------------------------------
En iyi model kaydedildi: /content/bigru_char_completion.pt
Epoch 02
Train Loss: 0.2050 | Train Char Acc: 0.9383 | Train Sent Acc: 0.0500
Dev   Loss: 0.1908 | Dev   Char Acc: 0.9423 | Dev   Sent Acc: 0.0623
--------------------------------------------------------------------------------
En iyi model kaydedildi: /content/bigru_char_completion.pt
Epoch 03
Train Loss: 0.1808 | Train Char Acc: 0.9460 | Train Sent Acc: 0.0701
Dev   Loss: 0.1736 | Dev   Char Acc: 0.9476 | Dev   Sent Acc: 0.0761
--------------------------------------------------------------------------------
En iyi model kaydedildi: /content/bigru_char_completion.pt
Epoch 04
Train Loss: 0.1636 | Train Char Acc: 0.9513 | Train Sent Acc: 0.0850
Dev   Loss: 0.1630 | Dev   Char Acc: 0.9518 | Dev   Sent Acc:

In [13]:
# ------------------------------------------------------------
# 9) TEST DEĞERLENDİRME
# ------------------------------------------------------------
if test_loader is not None:
    test_loss, test_char_acc, test_sent_acc = evaluate(model, test_loader)
    print("TEST SONUCU")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Char Acc: {test_char_acc:.4f}")
    print(f"Test Sent Acc: {test_sent_acc:.4f}")


TEST SONUCU
Test Loss: 0.1581
Test Char Acc: 0.9542
Test Sent Acc: 0.1007


In [14]:
# ------------------------------------------------------------
# 10) TAHMİN
# ------------------------------------------------------------
def predict_text(text, model, stoi, itos, max_len):
    text = normalize_input(text)
    ids = encode(text)

    if len(ids) > max_len:
        ids = ids[:max_len]

    original_len = len(ids)

    while len(ids) < max_len:
        ids.append(stoi[PAD_TOKEN])

    x = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        pred_ids = logits.argmax(dim=-1).squeeze(0).tolist()

    output_chars = []
    for i in range(original_len):
        original_char = text[i]
        pred_char = itos[pred_ids[i]]

        if original_char == "_":
            output_chars.append(pred_char)
        else:
            output_chars.append(original_char)

    return "".join(output_chars)


In [15]:
# ------------------------------------------------------------
# 11) ÖRNEK TAHMİNLER
# ------------------------------------------------------------
examples = [
    "_u oy_nculara d_ b_ze d_ g_vensinler",
    "b_n sana g_veniyorum",
    "_ug_n hava ç_k g_zel",
    "t_rkiye gü_el b_r ülk_d_r"
]

print("\nÖRNEK TAHMİNLER")
for ex in examples:
    pred = predict_text(ex, model, stoi, itos, max_len)
    print("input : ", ex)
    print("output: ", pred)
    print("-" * 80)


ÖRNEK TAHMİNLER
input :  _u oy_nculara d_ b_ze d_ g_vensinler
output:  bu oyunculara da bize de güvensinler
--------------------------------------------------------------------------------
input :  b_n sana g_veniyorum
output:  bun sana güveniyorum
--------------------------------------------------------------------------------
input :  _ug_n hava ç_k g_zel
output:  bugün hava çok gözel
--------------------------------------------------------------------------------
input :  t_rkiye gü_el b_r ülk_d_r
output:  türkiye günel bir ülkeder
--------------------------------------------------------------------------------
